![image_1781100254696.png](./image_1781100254696.png "image_1781100254696.png")

![image_1781100276494.png](./image_1781100276494.png "image_1781100276494.png")

![image_1781100298205.png](./image_1781100298205.png "image_1781100298205.png")

![image_1781100320114.png](./image_1781100320114.png "image_1781100320114.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window

# Initialize Spark session
spark = SparkSession.builder.appName("CustomersOrdersDF").getOrCreate()

# Customers dataset
customers_data = [
    (10, "Ivan Petrov", "Central"),
    (11, "Julia Tanaka", "Central"),
    (12, "Karl Jensen", "Central"),
    (13, "Lisa Andrade", "Pacific"),
    (14, "Mike O'Brien", "Pacific")
]

customers_columns = ["id", "name", "region"]

customers_df = spark.createDataFrame(customers_data, customers_columns)

# Orders dataset
orders_data = [
    (20, 10, 500),
    (21, 10, 500),
    (22, 11, 1200),
    (23, 12, 300),
    (24, 13, 2800),
    (25, 14, 2800)
]

orders_columns = ["id", "customer_id", "amount"]

orders_df = spark.createDataFrame(orders_data, orders_columns)

# Show DataFrames
print("Customers DataFrame:")
customers_df.show()

print("Orders DataFrame:")
orders_df.show()


In [0]:
result_df=(
    customers_df
    .join(orders_df,customers_df.id==orders_df.customer_id,"inner")
    .groupBy("name","region")
    .agg(
    f.sum(f.col("amount")).alias("total_spent")
    )
    .withColumn("rn",f.dense_rank()
                .over(
                    Window.partitionBy("region")
                    .orderBy(f.col("total_spent").desc(),f.col("name").asc())
                    )
    )
    .filter(f.col("rn")==1)
    .select(
        f.col("region"),
        f.col("name").alias("customer_name"),
        f.col("total_spent")
    )
)
display(result_df)
